## Body part clustering from BVH file

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.io import savemat
from clustering_bvh import process_all_files, derive_piece_list, BVH_DIR_DEFAULT, BVH_JOINTS

today = datetime.now().strftime("%d%b").lower()

# --- piece list -----------------------------------------------------------
# piece_list = derive_piece_list(BVH_DIR_DEFAULT)   # all files in bvh_to_csv_centered/
with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

modes = ["group", "individual", "audience"]

print(f"Processing {len(piece_list)} files, modes={modes}")
print(f"BVH joints ({len(BVH_JOINTS)}): {BVH_JOINTS}")

# --- Full-body extraction -------------------------------------------------
cluster_data = process_all_files(
    piece_list=piece_list,
    modes=modes,
    bvh_dir=BVH_DIR_DEFAULT,
    base_path_cycles="data/virtual_cycles",
    debug=True,
)

# --- Save -----------------------------------------------------------------
save_pkl = os.path.join("data", "cluster_data",
                         f"cluster_data_bvh_all_body_parts_{today}.pkl")
save_mat = os.path.join("data", "cluster_data",
                         f"cluster_data_bvh_all_body_parts_{today}.mat")

if cluster_data is not None:
    # with open(save_pkl, 'wb') as f:
    #     pickle.dump(cluster_data, f)
    print(f"[INFO] Saved pkl: {save_pkl}")

    # print("[INFO] Saving MAT file...")
    # savemat(save_mat, cluster_data)
    # print(f"[INFO] Saved mat: {save_mat}")

    print(f"[INFO] Summary:")
    print(f"  Total cycles  : {len(cluster_data['file_name'])}")
    print(f"  Files         : {len(set(cluster_data['file_name']))}")
    print(f"  Dance modes   : {set(cluster_data['dmode_name'])}")
    print(f"  BVH joints    : {list(cluster_data['body_parts'].keys())}")
else:
    print("[ERROR] process_all_files returned None")

## Build 7 x 3 x 2 x 101 cluster representation (real-time world coords -> cycle-phase grid)

Builds, **per cycle**, a `(7, 3, 2, 101)` array from the Nov-13 `cluster_data` above:

- **7 clusters** (index order): `COM, RL, LL, RH, LH, body, Head`
  - `COM` = `center_of_mass` **as-is**
  - `RL/LL` = mean of `Upper/Lower-Leg + Foot` (per side)
  - `RH/LH` = mean of `UpperArm + ForeArm + Hand` (per side)
  - `body` = mean of `Pelvis, T8`
  - `Head` = `Head` alone
- **3** = world `x, y, z` (meters, not phase-normalized)
- **2** = `[position=0, velocity=1]`
- **101** = cycle-phase grid `X = arange(0, 1.01, 0.01)`; `C(T) = (t - cycle_start)/(cycle_end - cycle_start)`, linear interp `F(C) -> F(X)`

Does **not** modify the original extraction. Saves to `data/cluster_data/cluster_7x3x2_<today>.pkl` and `.mat`.

In [ ]:
import pickle
from clustering_bvh import process_all_files, BVH_DIR_DEFAULT
from cluster_7x3x2_bvh import build_cluster_7x3x2, save_cluster_7x3x2, CLUSTER_MEMBERS

# Reuse in-memory cluster_data if available; otherwise rebuild from BVH CSVs.
try:
    cluster_data
    print("[INFO] Reusing in-memory cluster_data.")
except NameError:
    print("[INFO] cluster_data not in memory — rebuilding from BVH CSVs ...")
    with open('data/selected_piece_list.pkl', 'rb') as f:
        piece_list = pickle.load(f)
    cluster_data = process_all_files(
        piece_list=piece_list,
        modes=["group", "individual", "audience"],
        bvh_dir=BVH_DIR_DEFAULT,
        base_path_cycles="data/virtual_cycles",
        debug=False,
    )

print("[INFO] BVH cluster membership:")
for name, members in CLUSTER_MEMBERS.items():
    print(f"   {name:5s} <- {'center_of_mass (as-is)' if members is None else members}")

out = build_cluster_7x3x2(cluster_data, phase_step=0.01)

print(f"[INFO] data shape: {out['data'].shape}  (n_cycles, cluster, xyz, [pos,vel], phase)")
print(f"[INFO] phase grid: {out['phase_grid'][0]:.2f} .. {out['phase_grid'][-1]:.2f} "
      f"({len(out['phase_grid'])} points)")

paths = save_cluster_7x3x2(out, out_dir="data/cluster_data")
print("[INFO] Saved:", paths)

## Visualization of BVH cluster trajectories

**Data:** `(5966, 7, 3, 2, 101)` → `(n_cycles, cluster, xyz, [pos=0 / vel=1], phase)`  
**Coordinate system:** X mediolateral · Y vertical · Z anterior  
**Units:** cm (position) · cm/s (velocity)

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PKL = "data/cluster_data/cluster_7x3x2_bvh_05jun.pkl"
with open(PKL, "rb") as f:
    out = pickle.load(f)

data          = out["data"]           # (5966, 7, 3, 2, 101)
phase         = out["phase_grid"]     # (101,)  0 .. 1
cluster_names = out["cluster_names"]  # ['COM','RL','LL','RH','LH','body','Head']

pieces    = np.array(out["piece"])
dmode     = np.array(out["dmode_name"])
ensembles = np.array(out["ensemble"])

# axis / channel indices
XI, YI, ZI = 0, 1, 2
POS, VEL    = 0, 1

PIECE_COLORS = {
    "Suku":      "#2196F3",
    "Maraka":    "#FF9800",
    "Wasulunka": "#4CAF50",
    "Manjanin":  "#E91E63",
}
MODE_COLORS = {
    "group":      "#1565C0",
    "individual": "#E65100",
    "audience":   "#2E7D32",
}
ENSEMBLE_COLORS = {
    "E1": "#5C6BC0",
    "E2": "#EF6C00",
    "E3": "#388E3C",
}
DANCER_COLORS = {
    "dancer_1": "#9C27B0",
    "dancer_2": "#00BCD4",
    "dancer_3": "#FF5722",
    "dancer_4": "#8BC34A",
    "unknown":  "#9E9E9E",
}

# Derive dancer_id from ensemble + day (pkl predates the dancer_id field)
DANCER_MAP = {
    ("E1", "D1"): "dancer_1",
    ("E1", "D5"): "dancer_1",
    ("E1", "D2"): "dancer_2",
    ("E2", "D3"): "dancer_3",
    ("E2", "D4"): "dancer_3",
    ("E3", "D5"): "dancer_4",
    ("E3", "D6"): "dancer_4",
}
if "dancer_id" in out:
    dancers = np.array(out["dancer_id"])
else:
    dancers = np.array([
        DANCER_MAP.get((e, d), "unknown")
        for e, d in zip(out["ensemble"], out["day"])
    ])

def plot_mean_std(ax, phase, values, color, label=None, alpha_fill=0.2, lw=2):
    """Plot mean ± 1 SD band over the phase grid."""
    mu  = np.nanmean(values, axis=0)
    sd  = np.nanstd(values,  axis=0)
    ax.plot(phase, mu, color=color, lw=lw, label=label)
    ax.fill_between(phase, mu - sd, mu + sd, color=color, alpha=alpha_fill)

print(f"Loaded: {data.shape}")
print(f"Pieces: {sorted(set(pieces))} | Modes: {sorted(set(dmode))} | Ensembles: {sorted(set(ensembles))}")

In [ ]:
# ── Plot 1: Grand mean ± SD  –  all 7 clusters, Y (vertical) and Z (anterior) ──

fig, axes = plt.subplots(
    7, 2,
    figsize=(12, 16),
    sharex=True,
    gridspec_kw={"hspace": 0.05, "wspace": 0.25},
)

col_labels = ["Y  (vertical, cm)", "Z  (anterior, cm)"]
axis_indices = [YI, ZI]

for ci, cname in enumerate(cluster_names):
    for ai, (axis_idx, col_label) in enumerate(zip(axis_indices, col_labels)):
        ax = axes[ci, ai]
        values = data[:, ci, axis_idx, POS, :]   # (5966, 101)

        plot_mean_std(ax, phase, values, color="#1A237E")

        ax.axvline(0, color="k", lw=0.5, ls="--")
        ax.axvline(1, color="k", lw=0.5, ls="--")
        ax.set_ylabel(cname, fontsize=9, rotation=0, labelpad=28, va="center")
        ax.yaxis.set_label_position("left")

        if ci == 0:
            ax.set_title(col_label, fontsize=10, pad=6)
        if ci == len(cluster_names) - 1:
            ax.set_xlabel("Cycle phase", fontsize=9)

        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Grand mean ± 1 SD  |  all cycles  |  position", fontsize=12, y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: By piece  –  all clusters, Y and Z position ──

unique_pieces = sorted(set(pieces))

fig, axes = plt.subplots(
    7, 2,
    figsize=(12, 16),
    sharex=True,
    gridspec_kw={"hspace": 0.05, "wspace": 0.25},
)

for ci, cname in enumerate(cluster_names):
    for ai, (axis_idx, col_label) in enumerate(zip([YI, ZI], ["Y  (vertical, cm)", "Z  (anterior, cm)"])):
        ax = axes[ci, ai]
        for piece in unique_pieces:
            mask   = pieces == piece
            values = data[mask, ci, axis_idx, POS, :]
            plot_mean_std(ax, phase, values, color=PIECE_COLORS[piece], label=piece, lw=2)

        ax.set_ylabel(cname, fontsize=9, rotation=0, labelpad=28, va="center")
        if ci == 0:
            ax.set_title(col_label, fontsize=10, pad=6)
        if ci == len(cluster_names) - 1:
            ax.set_xlabel("Cycle phase", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

legend_handles = [
    Line2D([0], [0], color=PIECE_COLORS[p], lw=2, label=p) for p in unique_pieces
]
fig.legend(handles=legend_handles, loc="upper right", fontsize=9,
           bbox_to_anchor=(1.01, 1.0), title="Piece")
fig.suptitle("Mean ± 1 SD per piece  |  position", fontsize=12, y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: By dance mode  –  all clusters, Y and Z position ──

unique_modes = sorted(set(dmode))

fig, axes = plt.subplots(
    7, 2,
    figsize=(12, 16),
    sharex=True,
    gridspec_kw={"hspace": 0.05, "wspace": 0.25},
)

for ci, cname in enumerate(cluster_names):
    for ai, (axis_idx, col_label) in enumerate(zip([YI, ZI], ["Y  (vertical, cm)", "Z  (anterior, cm)"])):
        ax = axes[ci, ai]
        for mode in unique_modes:
            mask   = dmode == mode
            values = data[mask, ci, axis_idx, POS, :]
            plot_mean_std(ax, phase, values, color=MODE_COLORS[mode], label=mode, lw=2)

        ax.set_ylabel(cname, fontsize=9, rotation=0, labelpad=28, va="center")
        if ci == 0:
            ax.set_title(col_label, fontsize=10, pad=6)
        if ci == len(cluster_names) - 1:
            ax.set_xlabel("Cycle phase", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

legend_handles = [
    Line2D([0], [0], color=MODE_COLORS[m], lw=2, label=m) for m in unique_modes
]
fig.legend(handles=legend_handles, loc="upper right", fontsize=9,
           bbox_to_anchor=(1.01, 1.0), title="Dance mode")
fig.suptitle("Mean ± 1 SD per dance mode  |  position", fontsize=12, y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Left vs Right symmetry  –  RL/LL and RH/LH, all axes ──

RL_idx = cluster_names.index("RL")
LL_idx = cluster_names.index("LL")
RH_idx = cluster_names.index("RH")
LH_idx = cluster_names.index("LH")

pairs = [
    ("RL vs LL  (legs)",  RL_idx, "#D32F2F", LL_idx, "#1565C0"),
    ("RH vs LH  (arms)",  RH_idx, "#E65100", LH_idx, "#1B5E20"),
]

fig, axes = plt.subplots(
    len(pairs), 3,
    figsize=(14, 6),
    sharex=True,
    gridspec_kw={"hspace": 0.35, "wspace": 0.3},
)

for row, (title, idx_A, col_A, idx_B, col_B) in enumerate(pairs):
    labels_A = [cluster_names[idx_A], cluster_names[idx_B]]
    for ai, (axis_idx, axis_label) in enumerate(zip([XI, YI, ZI], ["X  (mediolateral)", "Y  (vertical)", "Z  (anterior)"])):
        ax = axes[row, ai]

        vals_A = data[:, idx_A, axis_idx, POS, :]
        vals_B = data[:, idx_B, axis_idx, POS, :]

        plot_mean_std(ax, phase, vals_A, color=col_A, label=cluster_names[idx_A], lw=2)
        plot_mean_std(ax, phase, vals_B, color=col_B, label=cluster_names[idx_B], lw=2)

        if row == 0:
            ax.set_title(axis_label, fontsize=10)
        if row == len(pairs) - 1:
            ax.set_xlabel("Cycle phase", fontsize=9)
        if ai == 0:
            ax.set_ylabel("Position (cm)", fontsize=9)

        ax.legend(fontsize=8, loc="upper right")
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

    axes[row, 1].set_title(f"{axes[row,1].get_title()}\n{title}", fontsize=10)

fig.suptitle("Left vs Right symmetry  |  mean ± 1 SD  |  position", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 6: Per dancer  –  all clusters, Y and Z position ──

unique_dancers = ["dancer_1", "dancer_2", "dancer_3", "dancer_4"]

fig, axes = plt.subplots(
    7, 2,
    figsize=(12, 16),
    sharex=True,
    gridspec_kw={"hspace": 0.05, "wspace": 0.25},
)

for ci, cname in enumerate(cluster_names):
    for ai, (axis_idx, col_label) in enumerate(zip(
        [YI, ZI], ["Y  (vertical, cm)", "Z  (anterior, cm)"]
    )):
        ax = axes[ci, ai]
        for dancer in unique_dancers:
            mask   = dancers == dancer
            if mask.sum() == 0:
                continue
            values = data[mask, ci, axis_idx, POS, :]
            plot_mean_std(ax, phase, values,
                          color=DANCER_COLORS[dancer], label=dancer, lw=2)

        ax.set_ylabel(cname, fontsize=9, rotation=0, labelpad=28, va="center")
        if ci == 0:
            ax.set_title(col_label, fontsize=10, pad=6)
        if ci == len(cluster_names) - 1:
            ax.set_xlabel("Cycle phase", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

legend_handles = [
    Line2D([0], [0], color=DANCER_COLORS[d], lw=2, label=d) for d in unique_dancers
]
fig.legend(handles=legend_handles, loc="upper right", fontsize=9,
           bbox_to_anchor=(1.01, 1.0), title="Dancer")
fig.suptitle("Mean ± 1 SD per dancer  |  position", fontsize=12, y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 7: KDE of hand-clap onsets by cycle phase ──
# Works with new pkl files that contain onset metadata, and also with older pkl files
# by falling back to the original onset CSVs.

from pathlib import Path
import pandas as pd
from scipy.stats import gaussian_kde


def _metadata_source():
    """Use out when available; otherwise use cluster_data."""
    if "out" in globals() and all(k in out for k in ["file_name", "cycle_start", "cycle_end"]):
        return out
    if "cluster_data" in globals() and all(k in cluster_data for k in ["file_name", "cycle_start", "cycle_end"]):
        return cluster_data
    raise KeyError("Need out or cluster_data in memory with file_name/cycle_start/cycle_end metadata.")


def _read_onset_csv(file_name, key):
    """Read absolute onset times for one recording and onset type."""
    if key == "hand_clap_onsets":
        path = Path("data/hand_clap_onsets_05jun2026") / f"{file_name}_hand_clap_onsets.csv"
        col = "contact_times"
    else:
        suffix = {
            "both_feet_onsets": "both_feet_onsets",
            "left_foot_onsets": "left_foot_onsets",
            "right_foot_onsets": "right_foot_onsets",
        }[key]
        path = (
            Path("data/dance_onsets_v4_0.007_foot_jun3")
            / f"{file_name}_T"
            / "onset_info"
            / f"{file_name}_T_{suffix}.csv"
        )
        col = "time_sec"

    if not path.exists():
        return np.array([], dtype=float)
    return pd.read_csv(path)[col].dropna().to_numpy(dtype=float)


_ONSET_CACHE = {}

def _get_cyclewise_onsets(key):
    """Return one onset-time array per cycle, falling back to source CSVs if needed."""
    src = _metadata_source()
    if key in src:
        return src[key]

    cyclewise = []
    file_names = src["file_name"]
    c_starts = np.asarray(src["cycle_start"], dtype=float)
    c_ends = np.asarray(src["cycle_end"], dtype=float)

    for file_name, c_start, c_end in zip(file_names, c_starts, c_ends):
        cache_key = (file_name, key)
        if cache_key not in _ONSET_CACHE:
            _ONSET_CACHE[cache_key] = _read_onset_csv(file_name, key)
        vals = _ONSET_CACHE[cache_key]
        cyclewise.append(vals[(vals >= c_start) & (vals <= c_end)])

    return cyclewise


def _onsets_to_phase(onsets_by_cycle, cycle_start, cycle_end):
    """Convert absolute onset times to cycle phase values in [0, 1]."""
    phase_values = []
    for onsets, c_start, c_end in zip(onsets_by_cycle, cycle_start, cycle_end):
        dur = c_end - c_start
        if dur <= 0:
            continue
        vals = np.asarray(onsets, dtype=float)
        if len(vals) == 0:
            continue
        phase_vals = (vals - c_start) / dur
        phase_values.extend(phase_vals[(phase_vals >= 0) & (phase_vals <= 1)])
    return np.asarray(phase_values, dtype=float)


def _plot_phase_kde(ax, phase_values, color, label, bw_method=0.08):
    """Plot a KDE over cycle phase with a rug plot of raw phase values."""
    x = np.linspace(0, 1, 400)
    if len(phase_values) >= 2:
        kde = gaussian_kde(phase_values, bw_method=bw_method)
        density = kde(x)
        ax.plot(x, density, color=color, lw=2.5, label=f"{label} (n={len(phase_values)})")
        ax.fill_between(x, density, color=color, alpha=0.2)
    elif len(phase_values) == 1:
        ax.axvline(phase_values[0], color=color, lw=2.5, label=f"{label} (n=1)")
    else:
        ax.plot([], [], color=color, lw=2.5, label=f"{label} (n=0)")

    if len(phase_values) > 0:
        ax.plot(phase_values, np.zeros_like(phase_values), "|", color=color, alpha=0.12, markersize=8)


src = _metadata_source()
cycle_start_arr = np.asarray(src["cycle_start"], dtype=float)
cycle_end_arr = np.asarray(src["cycle_end"], dtype=float)

hand_phase = _onsets_to_phase(
    _get_cyclewise_onsets("hand_clap_onsets"),
    cycle_start_arr,
    cycle_end_arr,
)

fig, ax = plt.subplots(figsize=(9, 4))
_plot_phase_kde(ax, hand_phase, color="#9C27B0", label="hand clap")
ax.set_xlim(0, 1)
ax.set_xlabel("Cycle phase")
ax.set_ylabel("Density")
ax.set_title("Hand-clap onset phase KDE")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 8: KDE of feet onsets by cycle phase ──
# Separate KDE curves for both-feet, left-foot, and right-foot onsets.

feet_onset_specs = [
    ("both_feet_onsets",  "both feet",  "#263238"),
    ("left_foot_onsets",  "left foot",  "#1565C0"),
    ("right_foot_onsets", "right foot", "#D32F2F"),
]

fig, ax = plt.subplots(figsize=(9, 4))

for key, label, color in feet_onset_specs:
    phase_vals = _onsets_to_phase(
        _get_cyclewise_onsets(key),
        np.asarray(out["cycle_start"], dtype=float),
        np.asarray(out["cycle_end"], dtype=float),
    )
    _plot_phase_kde(ax, phase_vals, color=color, label=label)

ax.set_xlim(0, 1)
ax.set_xlabel("Cycle phase")
ax.set_ylabel("Density")
ax.set_title("Feet onset phase KDE")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()